# Комп'ютерна графіка — система трьох проекцій

## Інтерактивний тренажер

Одна просторова точка має три взаємопов'язані проекції:

**A₁ = (X, Z)** — фронтальна  
**A₂ = (X, Y)** — горизонтальна  
**A₃ = (Y, Z)** — профільна

Проекційні зв'язки показують, які координати є спільними для відповідних зображень.

Запустіть **одну наступну code-комірку**.

> **Технічна примітка:** 3D-площина будується як полігон, тому `X = const`, `Y = const` і `Z = const` не спричиняють помилку триангуляції.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Checkbox
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# ============================================================
# КОМП'ЮТЕРНА ГРАФІКА
# Тренажер системи трьох взаємопов'язаних проекцій
# ============================================================

COLORS = {
    "A": "tab:red",
    "B": "tab:blue",
    "C": "tab:green"
}

NAMES = ["A", "B", "C"]


def make_points(Ax, Ay, Az, Bx, By, Bz, Cx, Cy, Cz):
    return np.array([
        [Ax, Ay, Az],
        [Bx, By, Bz],
        [Cx, Cy, Cz]
    ], dtype=float)


def classify_plane(P):
    normal = np.cross(P[1] - P[0], P[2] - P[0])

    if np.linalg.norm(normal) < 1e-9:
        return "⚠️ Площина не визначена: точки A, B, C колінеарні"

    if np.allclose(P[:, 2], P[0, 2]):
        return "Горизонтальна площина  •  Z = const"

    if np.allclose(P[:, 1], P[0, 1]):
        return "Фронтальна площина  •  Y = const"

    if np.allclose(P[:, 0], P[0, 0]):
        return "Профільна площина  •  X = const"

    return "Площина загального положення"


def common_limits(P):
    low = P.min(axis=0)
    high = P.max(axis=0)

    span = np.maximum(high - low, 4)
    margin = 0.20 * span

    return low - margin, high + margin


def draw_point(ax, x, y, label, color):
    ax.scatter(
        x, y,
        s=75,
        color=color,
        zorder=5
    )

    ax.annotate(
        label,
        (x, y),
        xytext=(7, 7),
        textcoords="offset points",
        fontsize=12,
        fontweight="bold"
    )


def draw_projection(
    ax,
    P,
    dims,
    labels,
    title,
    limits
):
    first, second = dims
    low, high = limits

    names = ["X", "Y", "Z"]

    ax.set_xlim(low[first], high[first])
    ax.set_ylim(low[second], high[second])
    ax.set_aspect("equal", adjustable="box")

    ax.set_xlabel(names[first])
    ax.set_ylabel(names[second])
    ax.set_title(title, fontweight="bold")
    ax.grid(True, alpha=0.25)

    Q = P[:, [first, second]]

    contour = [0, 1, 2, 0]

    ax.fill(
        Q[:, 0],
        Q[:, 1],
        alpha=0.10
    )

    ax.plot(
        Q[contour, 0],
        Q[contour, 1],
        color="black",
        linewidth=1.5
    )

    for i, label in enumerate(labels):
        draw_point(
            ax,
            Q[i, 0],
            Q[i, 1],
            label,
            COLORS[NAMES[i]]
        )


def draw_projection_links(
    ax_front,
    ax_horizontal,
    ax_profile,
    P,
    limits,
    show_links
):
    if not show_links:
        return

    low, high = limits

    # --------------------------------------------------------
    # Головний принцип:
    #
    # A1 = (X, Z)
    # A2 = (X, Y)
    # A3 = (Y, Z)
    #
    # Тому:
    #
    # X -> зв'язує A1 та A2
    # Y -> зв'язує A2 та A3
    # Z -> зв'язує A1 та A3
    #
    # У кожній проекції показуємо напрямні,
    # які мають однакову координату.
    # --------------------------------------------------------

    for i, name in enumerate(NAMES):
        color = COLORS[name]

        x, y, z = P[i]

        # X: фронтальна ↔ горизонтальна
        ax_front.axvline(
            x,
            color=color,
            linestyle="--",
            linewidth=1,
            alpha=0.35
        )

        ax_horizontal.axvline(
            x,
            color=color,
            linestyle="--",
            linewidth=1,
            alpha=0.35
        )

        # Y: горизонтальна ↔ профільна
        ax_horizontal.axhline(
            y,
            color=color,
            linestyle="--",
            linewidth=1,
            alpha=0.35
        )

        ax_profile.axvline(
            y,
            color=color,
            linestyle="--",
            linewidth=1,
            alpha=0.35
        )

        # Z: фронтальна ↔ профільна
        ax_front.axhline(
            z,
            color=color,
            linestyle="--",
            linewidth=1,
            alpha=0.35
        )

        ax_profile.axhline(
            z,
            color=color,
            linestyle="--",
            linewidth=1,
            alpha=0.35
        )


def draw_3d(ax, P, limits):
    low, high = limits

    ax.set_xlim(low[0], high[0])
    ax.set_ylim(low[1], high[1])
    ax.set_zlim(low[2], high[2])

    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")

    ax.set_title(
        "Допоміжний 3D-вигляд",
        fontweight="bold"
    )

    contour = [0, 1, 2, 0]

    ax.plot(
        P[contour, 0],
        P[contour, 1],
        P[contour, 2],
        color="black",
        linewidth=1.5
    )

    normal = np.cross(P[1] - P[0], P[2] - P[0])

    if np.linalg.norm(normal) > 1e-9:
        # Будуємо площину як полігон.
        # plot_trisurf тут НЕ використовуємо:
        # при X=const або Y=const його XY-триангуляція вироджується.
        polygon = Poly3DCollection(
            [P],
            alpha=0.12,
            edgecolor="black",
            linewidth=1.2
        )
        ax.add_collection3d(polygon)
    else:
        ax.text2D(
            0.05,
            0.92,
            "Площина не визначена",
            transform=ax.transAxes,
            fontweight="bold"
        )

    for i, name in enumerate(NAMES):
        ax.scatter(
            *P[i],
            s=75,
            color=COLORS[name]
        )

        ax.text(
            *P[i],
            "  " + name,
            fontweight="bold"
        )

    ax.view_init(
        elev=24,
        azim=-58
    )


def trainer(
    Ax=2, Ay=2, Az=2,
    Bx=8, By=3, Bz=6,
    Cx=5, Cy=8, Cz=4,
    show_links=True
):
    P = make_points(
        Ax, Ay, Az,
        Bx, By, Bz,
        Cx, Cy, Cz
    )

    limits = common_limits(P)

    fig = plt.figure(figsize=(16, 9))

    ax_front = fig.add_subplot(221)
    ax_profile = fig.add_subplot(222)
    ax_horizontal = fig.add_subplot(223)
    ax_3d = fig.add_subplot(224, projection="3d")

    draw_projection(
        ax_front,
        P,
        (0, 2),
        ["A₁", "B₁", "C₁"],
        "ФРОНТАЛЬНА ПРОЕКЦІЯ  XZ",
        limits
    )

    draw_projection(
        ax_horizontal,
        P,
        (0, 1),
        ["A₂", "B₂", "C₂"],
        "ГОРИЗОНТАЛЬНА ПРОЕКЦІЯ  XY",
        limits
    )

    draw_projection(
        ax_profile,
        P,
        (1, 2),
        ["A₃", "B₃", "C₃"],
        "ПРОФІЛЬНА ПРОЕКЦІЯ  YZ",
        limits
    )

    draw_projection_links(
        ax_front,
        ax_horizontal,
        ax_profile,
        P,
        limits,
        show_links
    )

    draw_3d(
        ax_3d,
        P,
        limits
    )

    fig.suptitle(
        classify_plane(P),
        fontsize=17,
        fontweight="bold"
    )

    plt.tight_layout()
    plt.show()


interact(
    trainer,

    Ax=FloatSlider(
        min=0, max=10, step=1, value=2,
        description="A: X",
        continuous_update=False
    ),

    Ay=FloatSlider(
        min=0, max=10, step=1, value=2,
        description="A: Y",
        continuous_update=False
    ),

    Az=FloatSlider(
        min=0, max=10, step=1, value=2,
        description="A: Z",
        continuous_update=False
    ),

    Bx=FloatSlider(
        min=0, max=10, step=1, value=8,
        description="B: X",
        continuous_update=False
    ),

    By=FloatSlider(
        min=0, max=10, step=1, value=3,
        description="B: Y",
        continuous_update=False
    ),

    Bz=FloatSlider(
        min=0, max=10, step=1, value=6,
        description="B: Z",
        continuous_update=False
    ),

    Cx=FloatSlider(
        min=0, max=10, step=1, value=5,
        description="C: X",
        continuous_update=False
    ),

    Cy=FloatSlider(
        min=0, max=10, step=1, value=8,
        description="C: Y",
        continuous_update=False
    ),

    Cz=FloatSlider(
        min=0, max=10, step=1, value=4,
        description="C: Z",
        continuous_update=False
    ),

    show_links=Checkbox(
        value=True,
        description="Показувати проекційні зв'язки"
    )
)


## Експеримент 1 — зміна X

Змініть тільки **A: X**.

Спостерігайте: A₁ і A₂ переміщуються, тому що обидві проекції містять X. A₃ не змінюється.

## Експеримент 2 — зміна Y

Змініть тільки **A: Y**.

A₂ і A₃ переміщуються. A₁ не змінюється.

## Експеримент 3 — зміна Z

Змініть тільки **A: Z**.

A₁ і A₃ переміщуються. A₂ не змінюється.

### Головне правило

**X → фронтальна + горизонтальна**  
**Y → горизонтальна + профільна**  
**Z → фронтальна + профільна**